# 01 · Bronze ingest

Loads raw data into bronze Delta tables exactly as received, with lineage columns added:

1. **Source files.** Each new CSV under the landing folder is appended to `bronze_<entity>`. Files already recorded in `etl_load_log` are skipped, so re-running is safe.
2. **Weather API.** Daily weather per sales region is pulled from Open-Meteo, starting the day after the last date already loaded (a per-region watermark).

CSV columns are kept as text in bronze. Typing and validation happen in silver.

In [ ]:
%run 00_utils

In [ ]:
BATCH_ID = new_batch_id()
INGESTED_AT = datetime.now(timezone.utc)
LOAD_LOG = "etl_load_log"

LOAD_LOG_SCHEMA = T.StructType([
    T.StructField("file_path", T.StringType()),
    T.StructField("entity", T.StringType()),
    T.StructField("row_count", T.LongType()),
    T.StructField("batch_id", T.StringType()),
    T.StructField("loaded_at", T.TimestampType()),
])


def with_lineage(df, source):
    return (
        df.withColumn("_source_file", F.lit(source))
          .withColumn("_batch_id", F.lit(BATCH_ID))
          .withColumn("_ingested_at", F.lit(INGESTED_AT))
    )

log(f"Batch {BATCH_ID}")

## Source files

In [ ]:
loaded = set()
if table_exists(LOAD_LOG):
    loaded = {r.file_path for r in spark.table(LOAD_LOG).select("file_path").collect()}

files = list_files(LANDING_ROOT)
new_files = [f for f in files if f not in loaded]
log(f"{len(files)} files in landing, {len(new_files)} new")

for path in new_files:
    entity = path.rsplit("/", 1)[-1][: -len(".csv")].lower()
    if entity not in ENTITIES:
        log(f"  skipped (unknown entity): {path}")
        continue

    df = spark.read.option("header", True).option("inferSchema", False).csv(path)
    df = with_lineage(df, path).cache()
    rows = df.count()
    append(df, f"bronze_{entity}")
    df.unpersist()

    # Record each file right after it lands, so a failure part-way through never reloads finished files.
    append(spark.createDataFrame([(path, entity, rows, BATCH_ID, INGESTED_AT)], LOAD_LOG_SCHEMA), LOAD_LOG)
    log(f"  {entity:<22} {rows:>8,} rows  <-  {path.rsplit('/', 2)[-2]}/{path.rsplit('/', 1)[-1]}")

## Weather API

In [ ]:
import requests

WEATHER_TABLE = "bronze_weather_daily"
DAILY_FIELDS = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"]
WEATHER_SCHEMA = T.StructType(
    [T.StructField("region", T.StringType()), T.StructField("city", T.StringType()), T.StructField("weather_date", T.StringType())]
    + [T.StructField(f, T.DoubleType()) for f in DAILY_FIELDS]
)


def fetch_weather(lat, lon, start, end, retries=3):
    """Daily weather for one location. Retries with backoff on network or server errors."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "daily": ",".join(DAILY_FIELDS),
        "temperature_unit": "fahrenheit",
        "precipitation_unit": "inch",
        "timezone": "auto",
    }
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(WEATHER_URL, params=params, timeout=60)
            response.raise_for_status()
            return response.json()["daily"]
        except Exception as error:
            if attempt == retries:
                raise
            wait = 2 ** attempt
            log(f"  attempt {attempt} failed ({error}); retrying in {wait}s")
            time.sleep(wait)


# Watermark: the latest date already in bronze, per region.
last_loaded = {}
if table_exists(WEATHER_TABLE):
    last_loaded = {
        r.region: r.last_date
        for r in spark.table(WEATHER_TABLE).groupBy("region").agg(F.max("weather_date").alias("last_date")).collect()
    }

end = (date.today() - timedelta(days=WEATHER_LAG_DAYS)).isoformat()
rows = []
for region, (city, lat, lon) in REGION_CITIES.items():
    start = WEATHER_START
    if region in last_loaded:
        start = (date.fromisoformat(last_loaded[region]) + timedelta(days=1)).isoformat()
    if start > end:
        log(f"  {region:<8} up to date")
        continue
    daily = fetch_weather(lat, lon, start, end)
    for i, day in enumerate(daily["time"]):
        values = [None if daily[f][i] is None else float(daily[f][i]) for f in DAILY_FIELDS]
        rows.append((region, city, day, *values))
    log(f"  {region:<8} {city:<13} {start} to {end}: {len(daily['time'])} days")

if rows:
    append(with_lineage(spark.createDataFrame(rows, WEATHER_SCHEMA), WEATHER_URL), WEATHER_TABLE)
log(f"Weather rows loaded: {len(rows):,}")